# FEM 2D Reference using GMSH Tools

This notebook serves as a reference for performing FEM analysis in 2D using `gmshtools.py` package from Prof. Weizenecker. It includes steps for creating a mesh, defining material properties, applying boundary conditions, and solving the FEM problem.

## Generating the Mesh
This section will cover how to generate a mesh using GMSH and prepare it for FEM analysis.

In [1]:
import numpy as np
import gmsh

from helper_funcs.colors import Colors as colors

from helper_funcs.gmshtools import ElementMsh, MshHs

In [2]:
gmsh.initialize()
name = "FEM_2D_testing"
gmsh.model.add(name)

# =======================
# create geometry
#   Add lines, points, surfaces, and holes to define the geometry.
# =======================
P1 = (1,1,0)
P1_5 = (2, 1, 0)
P2 = (3,1,0)
P3 = (1,4,0)
P3_5 = (2, 4, 0)
P4 = (3,4,0)

i1 = gmsh.model.occ.addPoint(*P1)
i1_5 = gmsh.model.occ.addPoint(*P1_5)
i2 = gmsh.model.occ.addPoint(*P2)
i3 = gmsh.model.occ.addPoint(*P3)
i3_5 = gmsh.model.occ.addPoint(*P3_5)
i4 = gmsh.model.occ.addPoint(*P4)

L1 = gmsh.model.occ.addLine(i1, i1_5)
L2 = gmsh.model.occ.addLine(i1_5, i2)
L3 = gmsh.model.occ.addLine(i2, i4)
L4 = gmsh.model.occ.addLine(i4, i3_5)
L5 = gmsh.model.occ.addLine(i3_5, i3)
L6 = gmsh.model.occ.addLine(i3, i1)

loop1 = gmsh.model.occ.addCurveLoop([L1, L2, L3, L4, L5, L6])

# Kreisloch
C2 = (2.5, 2.5, 0)
r2 = 0.3
circle2 = gmsh.model.occ.addCircle(*C2, r2)
loop2 = gmsh.model.occ.addCurveLoop([circle2])

# Fläche
surface = gmsh.model.occ.addPlaneSurface([loop1, loop2]) # loop2 wird von loop1 subtrahiert

# Embed the inner curve (circle)
C1 = (1.5, 1.75, 0)
r1 = 0.35
circle1 = gmsh.model.occ.addCircle(*C1, r1)

# embed rechteck
P5 = (1.25, 3, 0)
P6 = (1.75, 3, 0)
P7 = (1.25, 3.5, 0)
P8 = (1.75, 3.5, 0)
i5 = gmsh.model.occ.addPoint(*P5)
i6 = gmsh.model.occ.addPoint(*P6)
i7 = gmsh.model.occ.addPoint(*P7)
i8 = gmsh.model.occ.addPoint(*P8)

L5 = gmsh.model.occ.addLine(i5, i6)
L6 = gmsh.model.occ.addLine(i6, i8)
L7 = gmsh.model.occ.addLine(i8, i7)
L8 = gmsh.model.occ.addLine(i7, i5)

loop3 = gmsh.model.occ.addCurveLoop([L5, L6, L7, L8])

# embed line

L9 = gmsh.model.occ.addLine(i1_5, i3_5)

# =======================
# Synchronize
# =======================
gmsh.model.occ.synchronize()

# =======================
# Embed curves into surface
# =======================
gmsh.model.mesh.embed(1, [circle1], 2, surface) # Embed M1
gmsh.model.mesh.embed(1, [L5, L6, L7, L8], 2, surface) # Embed Rechteck
gmsh.model.mesh.embed(1, [L9], 2, surface) # embed Line


# =======================
# Synchronize
# =======================
gmsh.model.occ.synchronize()

# =======================
# Add physical groups
# =======================
pyhsical_surface = gmsh.model.addPhysicalGroup(2, [surface])
gmsh.model.setPhysicalName(2, pyhsical_surface, "MainSurface")

l0 = gmsh.model.addPhysicalGroup(1, [L1, L4]) # linke und untere Rand vom Rechteck
gmsh.model.setPhysicalName(1, l0, "DirichletBoundary")

l1 = gmsh.model.addPhysicalGroup(1, [L2, L3]) # rechte und obere Rand vom Rechteck
gmsh.model.setPhysicalName(1, l1, "RobinBoundary")


# =======================
# Synchronize
# =======================
gmsh.model.occ.synchronize()


# ========================
# Generate the mesh and save it
# ========================
gmsh.option.setNumber("Mesh.SaveAll", 1)
mesh = gmsh.model.mesh.generate(2)
gmsh.write(f"{name}.msh")

try:
    gmsh.fltk.run()
except:
    print("No FLTK GUI available, skipping visualization.")

netz2 = MshHs(gmsh.model)

gmsh.finalize()

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Line)
Info    : [ 20%] Meshing curve 3 (Line)
Info    : [ 30%] Meshing curve 4 (Line)
Info    : [ 40%] Meshing curve 5 (Line)
Info    : [ 40%] Meshing curve 6 (Line)
Info    : [ 50%] Meshing curve 7 (Circle)
Info    : [ 60%] Meshing curve 8 (Circle)
Info    : [ 70%] Meshing curve 9 (Line)
Info    : [ 70%] Meshing curve 10 (Line)
Info    : [ 80%] Meshing curve 11 (Line)
Info    : [ 90%] Meshing curve 12 (Line)
Info    : [100%] Meshing curve 13 (Line)
Info    : Done meshing 1D (Wall 0.000499834s, CPU 0.00071s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.00539772s, CPU 0.005175s)
Info    : 116 nodes 268 elements
Info    : Writing 'FEM_2D_testing.msh'...
Info    : Done writing 'FEM_2D_testing.msh'
-------------------------------------------------------
Version       : 4.15.2
License       : GNU General Public License
Build OS 

Fontconfig warning: using without calling FcInit()


## Extracting info from the mesh

In [3]:
plist = netz2.points
print(plist)

[[1.         1.         0.        ]
 [2.         1.         0.        ]
 [3.         1.         0.        ]
 [1.         4.         0.        ]
 [2.         4.         0.        ]
 [3.         4.         0.        ]
 [2.8        2.5        0.        ]
 [1.85       1.75       0.        ]
 [1.25       3.         0.        ]
 [1.75       3.         0.        ]
 [1.25       3.5        0.        ]
 [1.75       3.5        0.        ]
 [1.33333333 1.         0.        ]
 [1.66666667 1.         0.        ]
 [2.33333333 1.         0.        ]
 [2.66666667 1.         0.        ]
 [3.         1.33333333 0.        ]
 [3.         1.66666667 0.        ]
 [3.         2.         0.        ]
 [3.         2.33333333 0.        ]
 [3.         2.66666667 0.        ]
 [3.         3.         0.        ]
 [3.         3.33333333 0.        ]
 [3.         3.66666667 0.        ]
 [2.66666667 4.         0.        ]
 [2.33333333 4.         0.        ]
 [1.66666667 4.         0.        ]
 [1.33333333 4.         0.  